In [7]:
import os
from data.dataset import Device
import pandas as pd
import re

data_path = 'spectral_data'
expert_files = []
scan_corder_files = []
weeks = {}
labels = None
target_device = Device.SCAN_CODER


def extract_low_cost_label(leaf_data_dir):
    label = leaf_data_dir.split('/')[-2].replace('_', '').replace('scan', '')
    return label[:label.find('Ra')]

# come interface for weekly basis data extraction
def get_weeks_stats(week):
    df_week = df[df['week'] == week]
    return df_week


def get_week(path):
    match = re.search(r'week(\d+)', path.lower())
    if match:
        return int(match.group(1))
    raise ValueError(f"Week not found in path: {path}")
    
def populate_data_by_week(parent_dir, specimen_data):
    specimen_data_dir = os.path.join(parent_dir, specimen_data)
    # week = int(specimen_data_dir.split('/')[1][-1])
    week = get_week(specimen_data_dir)
    if week not in weeks:
        weeks[week] = {}
    label = None
    if 'calculations' in specimen_data:
        label = specimen_data.split('_')[0]
    else:
        label = specimen_data.split('.')[0]
    truncated_label = label[:label.find('Ra')]
    if '#' in truncated_label:
        truncated_label = truncated_label[:truncated_label.find('#')]
    if truncated_label not in weeks[week].keys():
        weeks[week][truncated_label] = {'raw': [], 'calculations': []}
    if 'cal' in specimen_data:
        weeks[week][truncated_label]['calculations'].append(specimen_data)
    else:
        weeks[week][truncated_label]['raw'].append(specimen_data)


for root in os.listdir(data_path):
    root_dir = os.path.join(data_path, root)
    if root_dir.endswith('.xlsx'):
        expert_files.append(root_dir)
        continue

    for device in os.listdir(root_dir):
        device_dir = os.path.join(root_dir, device)
        for data in os.listdir(device_dir):
            data_dir = os.path.join(device_dir, data)
            if target_device == Device.LOW_COST:
                if 'Reading' in data:
                    for reading in os.listdir(data_dir):
                        reading_dir = os.path.join(data_dir, reading)
                        for disease_class in os.listdir(reading_dir):
                            disease_class_dir = os.path.join(reading_dir, disease_class)
                            for specimen in os.listdir(disease_class_dir):
                                specimen_dir = os.path.join(disease_class_dir, specimen)
                                for leaf_data in os.listdir(specimen_dir):
                                    leaf_data_dir = os.path.join(specimen_dir, leaf_data)
                                    week = get_week(leaf_data_dir)
                                    if week not in weeks:
                                        weeks[week] = {}
                                    leaf_label = extract_low_cost_label(leaf_data_dir)
                                    if leaf_label not in weeks[week].keys():
                                        weeks[week][leaf_label] = {'raw': None, 'img': None}
                                    if leaf_data_dir.endswith('.jpg'):
                                        weeks[week][leaf_label]['img'] = leaf_data_dir
                                    else:
                                        weeks[week][leaf_label]['raw'] = leaf_data_dir
                                    
            elif target_device == Device.SCAN_CODER:
                if data_dir.endswith('.csv'):
                    week = get_week(data_dir)
                    if week not in weeks:
                        weeks[week] = None
                    weeks[week] = data_dir
                    scan_corder_files.append(data_dir)
            elif target_device == Device.BIO_SCIENCE:
                if not 'Reading' in data_dir and not data_dir.endswith('.csv'):
                    for disease_category in os.listdir(data_dir):
                        if disease_category.endswith('.json'):
                            continue
                        disease_category_file = os.path.join(data_dir, disease_category)
                        for reading in os.listdir(disease_category_file):
                            reading_dir = os.path.join(disease_category_file, reading)
                            if 'R' in reading:
                                for  point in os.listdir(reading_dir):
                                    point_dir = os.path.join(reading_dir, point)
                                    for specimen_data in os.listdir(point_dir):
                                        if specimen_data.endswith('.png'):
                                            continue
                                        populate_data_by_week(point_dir, specimen_data)
                            else:
                                for specimen_data in os.listdir(reading_dir):
                                    if specimen_data.endswith('.png'):
                                        continue
                                    # populate_data_by_week(point_dir, specimen_data)
                                    populate_data_by_week(reading_dir, specimen_data)





In [10]:
# wrapping for working with BIO SCIENCE spectrometer
if target_device == Device.BIO_SCIENCE:
    temp = {}
    for week in weeks.keys():
        samp_temp = {}
        for key in weeks[week].keys():
            raw_count = len(weeks[week][key]['raw'])
            calculation_count = len(weeks[week][key]['calculations'])
           
            samp_temp[key] = {
                'raw_count': raw_count, 
                'calculation_count': calculation_count,
            }
        temp[week] = samp_temp
    
    
    
    
    data = temp
    rows = []
    for week, labels in data.items():
        for label, values in labels.items():
            rows.append({
                'week': week,
                'label': label,
                'raw_count': values['raw_count'],
                'calculation_count': values['calculation_count'],
            })
    df = pd.DataFrame(rows)
    labels = list(get_weeks_stats(4)['label'])

# low cost data count
if target_device == Device.LOW_COST:
    sorted_weeks = dict(sorted(weeks.items()))
    
    temp = {}
    for week in sorted_weeks.keys():
        smp_temp = {}
        for key in sorted_weeks[week].keys():
            has_raw = 0
            has_img = 0
            if sorted_weeks[week][key]['raw'] is not None:
                has_raw += 1
            if sorted_weeks[week][key]['img'] is not None:
                has_img += 1
            smp_temp[key] = {'raw_count': has_raw, 'img_count': has_img}
        temp[week] = smp_temp
    
    rows = []
    for week, labels in temp.items():
        for label, values in labels.items():
            rows.append({
                'week': week,
                'label': label,
                'raw_count': values['raw_count'],
                'img_count': values['img_count']
            })
    df = pd.DataFrame(rows)
    labels = list(get_weeks_stats(5)['label'])


# come interface for weekly basis data extraction
def get_weeks_stats(week):
    df_week = df[df['week'] == week]
    return df_week


"""" 
TODO:
    On weekly basis, sort the crops in a proper order
    Randomized selection should be based on the unique key
    All computation on indexing can be peformed on the labels
"""


# data loader wrapper for the SCAN_CODER DEVICE

def read_data_by_week(week:int):
    df = pd.read_csv(weeks[week])
    labels = list(df['Sample ID'])
    return df, labels

if target_device == Device.SCAN_CODER:
    _, labels = read_data_by_week(4)

In [19]:
def _load_low_cost_files(weeks):
    temp_df_buffer = []
    for week in weeks.keys():
        pd_smp = pd.read_csv(weeks[week])
        pd_smp['week'] = week
        temp_df_buffer.append(pd_smp)
    return pd.concat(temp_df_buffer)


df = _load_low_cost_files(weeks)

In [27]:
list(df[df['week'] == 4]['Sample ID'])

['C2 CBB5 Rc 1g',
 'C2 CBB5 Rc 1b',
 'C2 CBB5 Rb 1g',
 'C2 CBB5 Rb 1b',
 'C2 CBB5 Ra 1g',
 'C2 CBB5 Ra 1b',
 'C2 CBB4 Rc 1g',
 'C2 CBB4 Rc 1b',
 'C2 CBB4 Rb 1g',
 'C2 CBB4 Rb 1b',
 'C2 CBB4 Ra 1g',
 'C2 CBB4 Ra 1b',
 'C2 CBB3 Rc 1g',
 'C2 CBB3 Rc 1b',
 'C2 CBB3 Rb 1g',
 'C2 CBB3 Rb 1b',
 'C2 CBB3 Ra 1g',
 'C2 CBB3 Ra 1b',
 'C2 CBB2 Rc 1g',
 'C2 CBB2 Rc 1b',
 'C2 CBB2 Rb 1g',
 'C2 CBB2 Rb 1b',
 'C2 CBB2 Ra 1g',
 'C2 CBB2 Ra 1b',
 'C2 CBB1 Rc 1g',
 'C2 CBB1 Rc 1b',
 'C2 CBB1 Rb 1g',
 'C2 CBB1 Rb 1b',
 'C2 CBB1 Ra 1g',
 'C2 CBB1 Ra 1b',
 'C2 CMD5 Rc 1g',
 'C2 CMD5 Rc 1b',
 'C2 CMD5 Rc 1b',
 'C2 CMD5 Rb 1g',
 'C2 CMD5 Rb 1b',
 'C2 CMD5 Ra 1g',
 'C2 CMD5 Ra 1b',
 'C2 CMD4 Rc 1g',
 'C2 CMD4 Rc 1b',
 'C2 CMD4 Rb 1g',
 'C2 CMD4 Rb 1b',
 'C2 CMD4 Ra 1g',
 'C2 CMD4 Ra 1b',
 'C2 CMD3 Rc 1g',
 'C2 CMD3 Rc 1b',
 'C2 CMD3 Rb 1g',
 'C2 CMD3 Rb 1b',
 'C2 CMD3 Ra 1g',
 'C2 CMD3 Ra 1b',
 'C2 CMD2 Rc 1g',
 'C2 CMD2 Rc 1b',
 'C2 CMD2 Rb 1g',
 'C2 CMD2 Rb 1b',
 'C2 CMD2 Ra 1g',
 'C2 CMD2 Ra 1b',
 'C2 CMD1 